# Download & Extract 500 PPTX Files from Zenodo10K

**Important correction from the first version of this notebook:** the `Forceless/Zenodo10K`
parquet split only contains *metadata* (filename, url, checksum, license, etc.) — it is not
storing the raw `.pptx` bytes inside the parquet. The actual files live as separate files
inside the same Hugging Face dataset repo, at paths like:

```
pptx/{license}/{year}/{checksum_without_prefix}-{filename}
```

So downloading is really: **(1)** load the metadata rows, **(2)** find each row's real
file path in the repo by matching its checksum, **(3)** download that exact file via
`hf_hub_download`. This is far more reliable than hitting the `url` column directly,
which points at Zenodo and can return `403 Forbidden` for scripted requests.

This notebook:
1. Installs required packages
2. Loads the (small, ~4MB) metadata split
3. Samples 500 rows
4. Gets the real list of files in the HF repo and matches each sample to its file by checksum
5. Downloads the matched files
6. Cleans the output (drops corrupt/empty files, dedupes by content hash)
7. Exports a manifest CSV
8. Sanity-checks a few files by opening them with `python-pptx`

Run cells top to bottom. Adjust `N_SAMPLES`, `OUTPUT_DIR`, and `RANDOM_SEED` in the config cell as needed.

In [5]:
# 1. Install dependencies (skip if already installed)
%pip install -q datasets huggingface_hub pandas tqdm python-pptx

Note: you may need to restart the kernel to use updated packages.


In [6]:
# 2. Imports
import os
import shutil
import random
import zipfile
import hashlib
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download

In [ ]:
# 3. Config — edit these as needed
REPO_ID        = "Forceless/Zenodo10K"
SPLIT          = "pptx"           # the dataset's split name
N_SAMPLES      = 100
RANDOM_SEED    = 42
OUTPUT_DIR     = Path("/Users/rohit/Documents/Git/Enterprise_RAG_Pipeline/data")
RAW_DIR        = OUTPUT_DIR / "downloaded_raw_files"       # downloaded files land here (via HF cache symlink or copy)
CLEAN_DIR      = OUTPUT_DIR / "raw"      # validated files copied here
MANIFEST_PATH  = OUTPUT_DIR / "manifest.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
random.seed(RANDOM_SEED)

print(f"Files will be saved under: {OUTPUT_DIR.resolve()}")

Files will be saved under: /Users/rohit/Documents/Git/Enterprise_RAG_Pipeline/data


## Step 1 — Load metadata and sample 500 rows

This split is tiny (a few MB), so we load it directly rather than streaming — that way
we can shuffle and sample normally with pandas.

In [13]:
# 4. Load metadata and take a random sample
ds = load_dataset(REPO_ID, split=SPLIT)
df = ds.to_pandas()
print(f"Total rows in metadata: {len(df)}")
print(df.columns.tolist())

sample_df = df.sample(n=min(N_SAMPLES, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Sampled {len(sample_df)} rows")
sample_df.head()

Total rows in metadata: 10448
['filename', 'size', 'url', 'license', 'title', 'created', 'updated', 'doi', 'checksum']
Sampled 100 rows


,filename,size,url,license,title,created,updated,doi,checksum
0,ERROR_presentation_4214.pptx,711778,https://zenodo.org/api/records/7140037/files/E...,cc-by-4.0,Failure Sources in Machine Learning for Medici...,2022-10-03T19:34:48.287395+00:00,2024-07-16T00:39:09.819123+00:00,10.5281/zenodo.7140037,md5:3927d8d158bfcda6916d4403122525ca
1,SCD UPHF_Ressources BU.pptx,3874166,https://zenodo.org/api/records/3254402/files/S...,cc-by-4.0,Support de formation - Services et ressources ...,2019-06-24T13:37:26.435771+00:00,2024-07-22T21:30:29.706212+00:00,10.5281/zenodo.3254402,md5:79c0c6010a2034c340955dbc862c1e1c
2,__sesion_5_calificadores_vision_general.pptx,5150208,https://zenodo.org/api/records/6950467/files/_...,cc-by-4.0,Indización de documentos LILACS 2022: califica...,2022-08-01T23:03:16.594590+00:00,2023-09-06T13:31:33.413315+00:00,10.5281/zenodo.6950467,md5:c09410080f883582814f54d5e3b6e8ec
3,uaw_2022.pptx,366673,https://zenodo.org/api/records/7417947/files/u...,cc-by-4.0,Transition from the IGS14 to IGS20 antenna mod...,2022-12-09T10:36:50.076206+00:00,2022-12-09T14:26:29.961738+00:00,10.5281/zenodo.7417947,md5:32e8db895514927d15002c5df788eb46
4,ESR12 - Sina DARBAN - Turino Oral Presentation...,5194087,https://zenodo.org/api/records/3572058/files/E...,cc-by-4.0,Investigation On Thermal Shock Resistance of A...,2019-12-12T11:03:35.347383+00:00,2024-07-22T15:29:09.926007+00:00,10.5281/zenodo.3572058,md5:7647ed05169d5b20d179ff721a7556d9


## Step 2 — Match each sampled row to its real file path in the repo

Every real file in the repo is named `{checksum_hash}-{filename}` (the `checksum` column
looks like `md5:abcdef...`, and the file path drops the `md5:` prefix). We list every
file in the repo once, then match samples to paths using that hash prefix — this avoids
having to guess the exact truncation/sanitization applied to long filenames.

In [14]:
# 5. List real repo files and build a lookup: hash_prefix -> full repo path
api = HfApi()
all_repo_files = api.list_repo_files(REPO_ID, repo_type="dataset")
pptx_repo_files = [f for f in all_repo_files if f.endswith(".pptx")]
print(f"Found {len(pptx_repo_files)} .pptx files in the repo")

# basename looks like: pptx/{license}/{year}/{hash}-{filename}.pptx
hash_to_path = {}
for path in pptx_repo_files:
    basename = path.rsplit("/", 1)[-1]
    if "-" in basename:
        hash_prefix = basename.split("-", 1)[0]
        hash_to_path[hash_prefix] = path

print(f"Built lookup with {len(hash_to_path)} entries")

Found 10448 .pptx files in the repo
Built lookup with 10393 entries


In [15]:
# 6. Match each sampled row to a real repo path via its checksum
def checksum_to_hash(checksum: str) -> str:
    # checksum format is like "md5:abcdef1234..." -> strip the algorithm prefix
    return checksum.split(":", 1)[-1] if ":" in checksum else checksum

matches = []
for i, row in sample_df.iterrows():
    h = checksum_to_hash(row["checksum"])
    repo_path = hash_to_path.get(h)
    matches.append({
        "index": i,
        "filename": row["filename"],
        "checksum": row["checksum"],
        "repo_path": repo_path,
        "matched": repo_path is not None,
    })

match_df = pd.DataFrame(matches)
print(f"Matched {match_df['matched'].sum()} / {len(match_df)} sampled rows to real files")
match_df[~match_df["matched"]].head()

Matched 100 / 100 sampled rows to real files


,index,filename,checksum,repo_path,matched


## Step 3 — Download the matched files

Uses `hf_hub_download`, which handles retries and caching for you. Files are fetched
into the local HF cache and then copied into `RAW_DIR` with their original filename.

In [16]:
# 7. Download each matched file
download_records = []

for _, row in tqdm(match_df[match_df["matched"]].iterrows(),
                    total=match_df["matched"].sum(), desc="Downloading"):
    try:
        cached_path = hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=row["repo_path"],
        )
        # sanitize filename before copying locally
        safe_name = "".join(c for c in str(row["filename"]) if c not in '\\/:*?"<>|').strip()
        if not safe_name.lower().endswith(".pptx"):
            safe_name += ".pptx"
        dest = RAW_DIR / safe_name
        if dest.exists():
            dest = RAW_DIR / f"{dest.stem}_{row['index']:04d}{dest.suffix}"
        shutil.copy2(cached_path, dest)
        download_records.append({
            "index": row["index"],
            "filename": dest.name,
            "size_bytes": dest.stat().st_size,
            "status": "downloaded",
        })
    except Exception as e:
        download_records.append({
            "index": row["index"],
            "filename": row["filename"],
            "status": f"download_error: {e}",
        })

# record the unmatched rows too, so the manifest accounts for all 500 samples
for _, row in match_df[~match_df["matched"]].iterrows():
    download_records.append({
        "index": row["index"],
        "filename": row["filename"],
        "status": "no_repo_match",
    })

download_df = pd.DataFrame(download_records)
print(download_df["status"].value_counts())

Downloading: 100%|██████████| 100/100 [04:20<00:00,  2.60s/it]

status
downloaded    100
Name: count, dtype: int64


## Step 4 — Clean the downloaded files

A `.pptx` is actually a ZIP archive. We validate each file by:
- Checking it's non-empty
- Checking it opens as a valid ZIP (catches truncated/corrupt downloads)
- Checking it contains `[Content_Types].xml`, which every real pptx has
- Deduping identical files by content hash

Valid files are copied into `CLEAN_DIR`; anything that fails is logged instead.

In [17]:
# 8. Validate and clean
def is_valid_pptx(path: Path) -> tuple[bool, str]:
    if path.stat().st_size == 0:
        return False, "empty_file"
    try:
        with zipfile.ZipFile(path) as z:
            names = z.namelist()
            if "[Content_Types].xml" not in names:
                return False, "missing_content_types"
    except zipfile.BadZipFile:
        return False, "bad_zip"
    return True, "ok"

seen_hashes = set()
clean_records = []

for path in tqdm(sorted(RAW_DIR.glob("*.pptx")), desc="Validating"):
    ok, reason = is_valid_pptx(path)
    if not ok:
        clean_records.append({"filename": path.name, "status": reason})
        continue

    file_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    if file_hash in seen_hashes:
        clean_records.append({"filename": path.name, "status": "duplicate"})
        continue
    seen_hashes.add(file_hash)

    dest = CLEAN_DIR / path.name
    shutil.copy2(path, dest)
    clean_records.append({
        "filename": path.name,
        "status": "clean",
        "size_bytes": path.stat().st_size,
        "sha256": file_hash,
    })

clean_df = pd.DataFrame(clean_records)
print(clean_df["status"].value_counts())
print(f"\n{ (clean_df['status'] == 'clean').sum() } clean files copied to {CLEAN_DIR}")

Validating: 100%|██████████| 174/174 [00:01<00:00, 116.58it/s]

status
clean        100
duplicate     74
Name: count, dtype: int64

100 clean files copied to /Users/rohit/Documents/Git/Enterprise_RAG_Pipeline/data/clean


## Step 5 — Export manifest

A CSV describing every sampled row: whether it was matched to a real file, downloaded,
and passed cleaning — useful for auditing the sample or re-running against just the
clean set.

In [18]:
# 9. Export manifest
manifest_df = match_df.merge(
    download_df[["index", "status"]].rename(columns={"status": "download_status"}),
    on="index", how="left"
).merge(
    clean_df.rename(columns={"filename": "final_filename"}),
    left_on="filename", right_on="final_filename", how="left", suffixes=("", "_clean")
)
manifest_df.to_csv(MANIFEST_PATH, index=False)

print(f"Manifest saved to: {MANIFEST_PATH.resolve()}")
manifest_df.head(10)

Manifest saved to: /Users/rohit/Documents/Git/Enterprise_RAG_Pipeline/data/manifest.csv


,index,filename,checksum,repo_path,matched,download_status,final_filename,status,size_bytes,sha256
0,0,ERROR_presentation_4214.pptx,md5:3927d8d158bfcda6916d4403122525ca,pptx/cc-by-4.0/2022/3927d8d158bfcda6916d440312...,True,downloaded,ERROR_presentation_4214.pptx,clean,711778.0,38c854ec91366c267c828b28b0af93062af5ae1ddc01b3...
1,1,SCD UPHF_Ressources BU.pptx,md5:79c0c6010a2034c340955dbc862c1e1c,pptx/cc-by-4.0/2019/79c0c6010a2034c340955dbc86...,True,downloaded,SCD UPHF_Ressources BU.pptx,clean,3874166.0,a9e0b730aa84b6e45cf890fb0defdd4f21ecc6c5660f14...
2,2,__sesion_5_calificadores_vision_general.pptx,md5:c09410080f883582814f54d5e3b6e8ec,pptx/cc-by-4.0/2022/c09410080f883582814f54d5e3...,True,downloaded,__sesion_5_calificadores_vision_general.pptx,clean,5150208.0,d2084636d109152e5c2d112d1114dbc3f6c47742ca75d2...
3,3,uaw_2022.pptx,md5:32e8db895514927d15002c5df788eb46,pptx/cc-by-4.0/2022/32e8db895514927d15002c5df7...,True,downloaded,uaw_2022.pptx,clean,366673.0,23ec40495a30b9c20c149fcb8063aaa09c2d123399fa36...
4,4,ESR12 - Sina DARBAN - Turino Oral Presentation...,md5:7647ed05169d5b20d179ff721a7556d9,pptx/cc-by-4.0/2019/7647ed05169d5b20d179ff721a...,True,downloaded,ESR12 - Sina DARBAN - Turino Oral Presentation...,clean,5194087.0,2b1c7dd6e513c5ee09542027c0e3285f2ca8e94d7a75ac...
5,5,(23-10-04) CIESIN Presentation - Mead.pptx,md5:8a0de41fc26bf2b84c93cd0d01c9ea34,pptx/cc-by-4.0/2023/8a0de41fc26bf2b84c93cd0d01...,True,downloaded,(23-10-04) CIESIN Presentation - Mead.pptx,clean,1058954.0,ef6d754e44c72531c30b3215c082d2558d7584ee8467e9...
6,6,ReproducibleComputationalEnvironment.pptx,md5:683ab66729608c6465435047c6eb8bab,pptx/cc-by-4.0/2020/683ab66729608c6465435047c6...,True,downloaded,ReproducibleComputationalEnvironment.pptx,clean,88174207.0,b1d31c572d28537b4b6b8c13e80f26e619f6031b32a6d2...
7,7,6. Attention models.pptx,md5:3082db51e29a6878599655698b418b4a,pptx/cc-by-4.0/2023/3082db51e29a6878599655698b...,True,downloaded,6. Attention models.pptx,clean,11533704.0,15706ba83046a05f6199d8c00e5ef760b6805fc4222e5f...
8,8,ReproHack-ICCS_TTW.pptx,md5:b36f134a5111b9c77320d841e859b372,pptx/cc-by-4.0/2024/b36f134a5111b9c77320d841e8...,True,downloaded,ReproHack-ICCS_TTW.pptx,clean,18639706.0,97a2f423cfd06fcb2e24375ec985774ce7ef4548a905fa...
9,9,IPPOG-Poster-20211231.pptx,md5:e452f42ea3804c2b1a63793a9b5f740e,pptx/cc-by-4.0/2022/e452f42ea3804c2b1a63793a9b...,True,downloaded,IPPOG-Poster-20211231.pptx,clean,3180155.0,0ba09651b893dd0297614a2266668d542741305fdeb2ea...


## Step 6 — Quick sanity check with `python-pptx`

Open a handful of the cleaned files to confirm they parse correctly before running your
full extraction pipeline against them.

In [19]:
# 10. Sanity check a few files
from pptx import Presentation

sample_files = list(CLEAN_DIR.glob("*.pptx"))[:5]
for f in sample_files:
    try:
        prs = Presentation(f)
        print(f"{f.name}: {len(prs.slides)} slides — OK")
    except Exception as e:
        print(f"{f.name}: FAILED to open — {e}")

ERROR_presentation_4214.pptx: 19 slides — OK
Design of Coal Handling .pptx: 21 slides — OK
uaw_2022.pptx: 13 slides — OK
5xPRO_T3_29.5.2024_NIB_PPT_ARIS.pptx: 25 slides — OK
chrisdone_padua18_final.pptx: 37 slides — OK


## Summary

- Raw downloaded files: `./pptx_data/raw/`
- Cleaned/validated files: `./pptx_data/clean/`  ← use this folder for your extraction pipeline
- Manifest: `./pptx_data/manifest.csv`

**Troubleshooting:**
- If `list_repo_files` in Step 2 is slow (~10K files), that's expected — it's a single API
  call, just a big response.
- If a lot of rows show `no_repo_match`, the checksum-matching logic may need adjusting —
  print a few entries of `pptx_repo_files` and `sample_df['checksum']` to compare formats.
- If you hit Hugging Face rate limits on repeated runs, set an HF token via
  `huggingface-cli login` first — authenticated requests get higher limits.
- Want more than 500 files? Just bump `N_SAMPLES` and re-run from Step 1; already-downloaded
  files are cached locally by `hf_hub_download` and won't be re-fetched.